In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, DataCollatorForTokenClassification, TrainingArguments, Trainer
from datasets import Dataset, Features, Sequence, Value, ClassLabel
from preprocessing import read_conll_file
import torch
import numpy as np
import evaluate

In [ ]:
en_train_sentences = read_conll_file(path="/content/drive/MyDrive/Colab Notebooks/Info_Extr_HW5_Project/data/eng_train.txt",
                                delimiter=" ",
                                langugage="en")

en_dev_sentences = read_conll_file(path="/content/drive/MyDrive/Colab Notebooks/Info_Extr_HW5_Project/data/eng_dev.txt",
                                delimiter=" ",
                                langugage="en")

de_train_sentences = read_conll_file(path="/content/drive/MyDrive/Colab Notebooks/Info_Extr_HW5_Project/data/deu_train.txt",
                                delimiter=" ",
                                langugage="de")

de_dev_sentences = read_conll_file(path="/content/drive/MyDrive/Colab Notebooks/Info_Extr_HW5_Project/data/deu_dev.txt",
                                delimiter=" ",
                                langugage="de")

de_test_sentences = read_conll_file(path="/content/drive/MyDrive/Colab Notebooks/Info_Extr_HW5_Project/data/deu_test.txt", delimiter=" ", langugage="de")

en_test_sentences = read_conll_file(path="/content/drive/MyDrive/Colab Notebooks/Info_Extr_HW5_Project/data/eng_test.txt", delimiter=" ", langugage="en")


en_de_concat_sentences = en_train_sentences + de_train_sentences

In [ ]:
for dataset in [de_train_sentences, de_dev_sentences, de_test_sentences]:
  for sentence in dataset[:10]:
    print(sentence.tokens)
    print(sentence.labels)
  print("------")
# print(de_test_sentences[:10]['labels'])


In [ ]:
en_train_processed = [sentence.to_dict() for sentence in en_train_sentences]
en_dev_processed = [sentence.to_dict() for sentence in en_dev_sentences]

de_train_processed = [sentence.to_dict() for sentence in de_train_sentences]
de_dev_processed = [sentence.to_dict() for sentence in de_dev_sentences]
de_test_processed = [sentence.to_dict() for sentence in de_test_sentences]

en_test_processed = [sentence.to_dict() for sentence in en_test_sentences]

In [ ]:

label_names = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC", "B-MISC", "I-MISC"]

features = Features({
    "tokens": Sequence(Value("string")),
    "labels": Sequence(ClassLabel(names=label_names))
})

en_train_set = Dataset.from_list(en_train_processed, features=features)
en_dev_set = Dataset.from_list(en_dev_processed, features=features)
en_test_set = Dataset.from_list(en_test_processed, features=features)

de_train_set = Dataset.from_list(de_train_processed, features=features)
de_dev_set = Dataset.from_list(de_dev_processed, features=features)
de_test_set = Dataset.from_list(de_test_processed, features=features)

en_de_concatenated_train_set = Dataset.from_list(en_train_processed + de_train_processed, features=features)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-base")

In [ ]:
def make_tokenize_fn(tokenizer):
    def tokenize_and_align(examples):
        tokenized = tokenizer(examples["tokens"],
                              is_split_into_words=True,
                              truncation=True)


        ### align labels with subwords
        all_labels = []
        # loop over sentences in batch
        for i in range(len(examples['labels'])):
            # IDs of which word each subword belongs to e.g. [None, 0, 1, 1, 2, 2, 3, None]
            word_ids = tokenized.word_ids(batch_index=i)
            labels_in_sentence = []
            prev_word_id = None
            for word_id in word_ids:
                # append label or -100 (if subword continuation or special token)
                if word_id==None or word_id==prev_word_id:
                    labels_in_sentence.append(-100)
                else:
                    label = examples['labels'][i][word_id]
                    labels_in_sentence.append(label)
                prev_word_id = word_id
            all_labels.append(labels_in_sentence)
        tokenized['labels'] = all_labels
        return tokenized
    return tokenize_and_align

In [ ]:
tokenize_fn = make_tokenize_fn(tokenizer)
tokenized_en_train_set = en_train_set.map(tokenize_fn, remove_columns=['tokens', 'labels'], batched=True)
tokenized_en_dev_set = en_dev_set.map(tokenize_fn, remove_columns=['tokens', 'labels'], batched=True)
tokenized_en_test_set = en_test_set.map(tokenize_fn, remove_columns=['tokens', 'labels'], batched=True)

tokenized_de_train_set = de_train_set.map(tokenize_fn, remove_columns=['tokens', 'labels'], batched=True)
tokenized_de_dev_set = de_dev_set.map(tokenize_fn, remove_columns=['tokens', 'labels'], batched=True)
tokenized_de_test_set = de_test_set.map(tokenize_fn, remove_columns=['tokens', 'labels'], batched=True)

tokenized_concatenated_train_set = en_de_concatenated_train_set.map(tokenize_fn, remove_columns=['tokens', 'labels'], batched=True)




In [ ]:
for item in tokenized_en_train_set:
    print(item['labels'])

In [ ]:

id2label = {i: label for i, label in enumerate(label_names)}
label2id = {v: k for k, v in id2label.items()}


In [ ]:

model_checkpoint = "FacebookAI/xlm-roberta-base"
finetuned_model = AutoModelForTokenClassification.from_pretrained(model_checkpoint, id2label=id2label, label2id=label2id)


In [ ]:
def freeze_all_but_n_top_layers(model, n=1):                                                                                                    
    for param in model.base_model.parameters():                                                              
        param.requires_grad = False                                                                          
    for layer in model.base_model.encoder.layer[-n:]:                                                        
        for param in layer.parameters():                                                                     
            param.requires_grad = True

In [ ]:
!pip install seqeval

In [ ]:
metric = evaluate.load("seqeval")

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    # convert to labels except special tokens (-100)
    true_labels = [[label_names[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    all_metrics = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": all_metrics["overall_precision"],
        "recall": all_metrics["overall_recall"],
        "f1": all_metrics["overall_f1"],
        "accuracy": all_metrics["overall_accuracy"]
    }

In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer)

In [ ]:
args = TrainingArguments(
    output_dir="/content/drive/MyDrive/Colab Notebooks/Info_Extr_HW5_Project/src/xlm-roberta-ner",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    seed=1
)



trainer = Trainer(
    model=finetuned_model,
    args=args,
    train_dataset=tokenized_de_train_set, # DE
    eval_dataset=tokenized_de_dev_set, # Dev set DE
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
)

In [ ]:
freeze_all_but_n_top_layers(finetuned_model)
trainer.train(resume_from_checkpoint=False)

In [ ]:
output = trainer.predict(tokenized_de_test_set)
logits, labels = output.predictions, output.label_ids
predictions = np.argmax(logits, axis=-1)

true_labels = [[label_names[l] for l in label if l != -100] for label in labels]
true_predictions = [
    [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]
print(f"true_labels: {true_labels}")
print(f"predicted: {true_predictions}")




In [ ]:
metrics = trainer.evaluate(tokenized_de_test_set)

In [ ]:
print("DE TEST RESULTS")
print(f"precision: {metrics["eval_precision"]}")
print(f"recall: {metrics["eval_recall"]}")
print(f"f1: {metrics["eval_f1"]}")

### add CRF Head

In [ ]:
!pip install torch

In [ ]:
# prepare dataloaders for batching
from torch.utils.data import DataLoader

# set random seed for reproducibility
seed = 1
g = torch.Generator()
g.manual_seed(seed)

train_en_dataloader = DataLoader(
    tokenized_en_train_set,
    batch_size=16,
    shuffle=True,
    collate_fn=data_collator,
    generator=g
)

train_de_dataloader = DataLoader(
    tokenized_de_train_set,
    batch_size=16,
    shuffle=True,
    collate_fn=data_collator,
    generator=g
)

train_concat_dataloader = DataLoader(
    tokenized_concatenated_train_set,
    batch_size=16,
    shuffle=True,
    collate_fn=data_collator,
    generator=g
)

dev_en_dataloader = DataLoader(
    tokenized_en_dev_set,
    batch_size=16,
    shuffle=False,
    collate_fn=data_collator,
)

dev_de_dataloader = DataLoader(
    tokenized_de_dev_set,
    batch_size=16,
    shuffle=False,
    collate_fn=data_collator,
)

test_en_dataloader = DataLoader(
    tokenized_en_test_set,
    batch_size=16,
    shuffle=False,
    collate_fn=data_collator,
)

test_de_dataloader = DataLoader(
    tokenized_de_test_set,
    batch_size=16,
    shuffle=False,
    collate_fn=data_collator,
)





In [ ]:
!pip install pytorch-crf

In [ ]:
from torchcrf import CRF


def initialize_crf(model):
    # put model on GPU if available
    if torch.cuda.is_available():
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")
    model.to(device)

    # freeze encoder parameters
    for param in model.base_model.parameters():
        param.requires_grad = False

    # new head
    num_tags = len(label_names)
    crf = CRF(num_tags, batch_first=True) # batch_first -> HF standard output dims (batch_size, sequence_length, num_tags)
    crf.to(device)

    

    # initialize impossible BIO transitions with very negative weights
    with torch.no_grad():
        for from_label, from_id in label2id.items():
            for to_label, to_id in label2id.items():
                if to_label.startswith('I-'):
                    entity_type = to_label[2:]
                    valid_preceding = [f"B-{entity_type}", f"I-{entity_type}"]
                    if from_label not in valid_preceding:
                        crf.transitions[from_id, to_id] = -10000
    
    return crf


def unfreeze_n_layers(model, n=1):                                                                                                    
    for param in model.base_model.parameters():                                                              
        param.requires_grad = False                                                                          
    for layer in model.base_model.encoder.layer[-n:]:                                                        
        for param in layer.parameters():                                                                     
            param.requires_grad = True
                                                                                                            
    optimizer = AdamW([                                                                                       
        {'params': [p for layer in model.base_model.encoder.layer[-n:] for p in layer.parameters()], 'lr':
    2e-5},                                                                                                   
        {'params': model.classifier.parameters(), 'lr': 2e-5, 'weight_decay':0.01},                                             
        {'params': crf.parameters(), 'lr': 1e-3},                                              
    ])

    return optimizer                                                                                                       
       

def word_level_gather(emissions, labels):
    # pack first-subword positions (labels != -100) into a word-level tensor
    # so the CRF scores one tag per word
    B = emissions.shape[0]
    emissions_per_row = []
    labels_per_row = []
    for b in range(B):
        is_first = labels[b] != -100
        emissions_per_row.append(emissions[b][is_first])
        labels_per_row.append(labels[b][is_first])

    word_emissions = torch.nn.utils.rnn.pad_sequence(emissions_per_row, batch_first=True)
    word_labels = torch.nn.utils.rnn.pad_sequence(labels_per_row, batch_first=True)

    max_words = word_emissions.shape[1]
    n_words = torch.tensor([t.shape[0] for t in labels_per_row], device=labels.device)
    word_mask = torch.arange(max_words, device=labels.device).unsqueeze(0) < n_words.unsqueeze(1)

    return word_emissions, word_labels, word_mask

In [ ]:
# CRF training step
def training_step(input_ids, attention_mask, labels, model):
    # emissions = logits from finetuned XLM-R
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    emissions = outputs.logits

    # collapse subword emissions to one position per word
    word_emissions, word_labels, word_mask = word_level_gather(emissions, labels)

    loss = -crf(word_emissions, word_labels, mask=word_mask, reduction='mean')
    return loss

In [ ]:
# CRF inference step
def predict(input_ids, attention_mask, labels, model):
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    emissions = outputs.logits

    word_emissions, _, word_mask = word_level_gather(emissions, labels)
    # crf.decode returns List[List[int]]... inner list length = each row's word count
    return crf.decode(word_emissions, mask=word_mask)

In [ ]:
# TRAINING LOOP
from torch.optim import Adam, AdamW

def run_training(train_dataloader, dev_dataloader, model, optimizer, num_epochs=3):
    # put model on GPU if available
    if torch.cuda.is_available():
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")
    model.to(device)

    for epoch in range(num_epochs):
        # training
        model.train()
        total_train_loss = 0.0
        for batch in train_dataloader:
            input_ids = batch['input_ids']
            attention_mask = batch['attention_mask']
            labels = batch['labels']

            # move batch to device
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device)

            # forward
            optimizer.zero_grad()
            loss = training_step(model=model, input_ids=input_ids,
                                 attention_mask=attention_mask,
                                 labels=labels
            )

            # backward
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()
        
        avg_train_loss = total_train_loss / len(train_dataloader)
        
        # validation
        with torch.no_grad():
            model.eval()
            total_val_loss = 0.0
            for batch in dev_dataloader:
                input_ids = batch['input_ids']
                attention_mask = batch['attention_mask']
                labels = batch['labels']

                # move batch to device
                input_ids = input_ids.to(device)
                attention_mask = attention_mask.to(device)
                labels = labels.to(device)

                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                emissions = outputs.logits

                word_emissions, word_labels, word_mask = word_level_gather(emissions, labels)

                loss = -crf(word_emissions, word_labels, mask=word_mask, reduction='mean')
                total_val_loss += loss.item()
        
        avg_val_loss = total_val_loss / len(dev_dataloader)
        
        print(f"epoch: {epoch + 1:>15}")
        print(f"training loss: {avg_train_loss:>15.2f}")
        print(f"validation loss: {avg_val_loss:>15.2f}")
        print("-" * 17)






In [ ]:
# PREDICT ON TEST SET
def evaluate_crf(test_dataloader, model):
    # put model on GPU if available
    if torch.cuda.is_available():
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")
    model.to(device)
    
    all_preds = []
    all_trues = []

    model.eval()
    with torch.no_grad():
        for batch in test_dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            # word-level predictions: List[List[int]] one tag per word
            pred_word_ids = predict(input_ids=input_ids,
                                    attention_mask=attention_mask,
                                    labels=labels,
                                    model=model)

            # word-level true labels: filter out -100s
            true_word_ids = [
                [int(l) for l in row.tolist() if l != -100]
                for row in labels
            ]

            for pred, true in zip(pred_word_ids, true_word_ids):
                assert len(pred) == len(true), \
                    f"length mismatch: pred {len(pred)} vs true {len(true)}"
                all_preds.append([label_names[p] for p in pred])
                all_trues.append([label_names[t] for t in true])

    metric = evaluate.load("seqeval")
    all_metrics = metric.compute(predictions=all_preds, references=all_trues)
    return {
        "precision": all_metrics["overall_precision"],
        "recall": all_metrics["overall_recall"],
        "f1": all_metrics["overall_f1"],
        "accuracy": all_metrics["overall_accuracy"]
    }


### Train CRF on top of finetuned XLM-R with frozen encoder

In [ ]:
crf = initialize_crf()

transitions_before = crf.transitions.detach().clone()
# optimizer = Adam at 1e-3
run_training(train_de_dataloader, dev_de_dataloader, finetuned_model, optimizer)

diff = (crf.transitions.detach() - transitions_before).abs()                                             
print(f"max change:  {diff.max().item():.4f}")
print(f"mean change: {diff.mean().item():.4f}")    

### Train CRF jointly with XLM-R base and n unfrozen top layers

In [ ]:
model_checkpoint = "FacebookAI/xlm-roberta-base"
base_model = AutoModelForTokenClassification.from_pretrained(model_checkpoint, id2label=id2label, label2id=label2id)
crf = initialize_crf(base_model)
optimizer = unfreeze_n_layers(base_model, n=1)

transitions_before = crf.transitions.detach().clone() 


run_training(train_de_dataloader, dev_de_dataloader, base_model, optimizer)

diff = (crf.transitions.detach() - transitions_before).abs()                                             
print(f"max change:  {diff.max().item():.4f}")
print(f"mean change: {diff.mean().item():.4f}")  

### Evaluate

In [ ]:
metrics = evaluate_crf(test_de_dataloader, base_model)

In [ ]:
metrics.keys()

In [ ]:
print("CRF jointly trained with XLM-R base (top 3 layers unfrozen) -- DE TEST RESULTS")
print(f"accuracy: {metrics["accuracy"]}")
print(f"precision: {metrics["precision"]}")
print(f"recall: {metrics["recall"]}")
print(f"f1: {metrics["f1"]}")


